<a href="https://colab.research.google.com/github/syahidmid/google-colab/blob/main/wordpress/tag/delete_tagipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth

# Jika menggunakan Google Colab
try:
    from google.colab import files, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    userdata = None

# ===============================
# CONFIG
# ===============================
wordpress_base_url = "https://www.balitouristic.com//wp-json/wp/v2/tags"

# Ambil kredensial
if userdata:
    username = userdata.get('admin_bali')
    password = userdata.get('pass_bali')
else:
    username = None
    password = None

if not username or not password:
    raise ValueError("Kredensial tidak ditemukan di userdata.")

auth = HTTPBasicAuth(username, password)

# ===============================
# UPLOAD CSV
# ===============================
if IN_COLAB:
    print("📤 Silakan upload file CSV...")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
else:
    filename = input("Masukkan path file CSV: ")

df = pd.read_csv(filename)

print("\n📊 Kolom terdeteksi:", df.columns.tolist())

# ===============================
# DETECT ID COLUMN
# ===============================
id_column = None
for col in df.columns:
    if col.strip().lower() == "id":
        id_column = col
        break

if not id_column:
    raise ValueError("Kolom 'ID' tidak ditemukan di CSV.")

print(f"✅ Menggunakan kolom '{id_column}' sebagai ID.\n")

tag_ids = df[id_column].dropna().astype(int).tolist()

print(f"🎯 Total ID terdeteksi: {len(tag_ids)}")

# ===============================
# CONFIRM DELETE
# ===============================
confirm = input("⚠ Yakin ingin menghapus tag tersebut? (yes/no): ")

if confirm.lower() != "yes":
    print("❌ Proses dibatalkan.")
else:
    print("\n🚀 Memulai proses delete...\n")

    deleted = 0
    failed = 0

    for tag_id in tag_ids:
        delete_url = f"{wordpress_base_url}/{tag_id}?force=true"

        response = requests.delete(delete_url, auth=auth)

        if response.status_code == 200:
            print(f"✅ Tag ID {tag_id} berhasil dihapus.")
            deleted += 1
        else:
            print(f"❌ Gagal hapus Tag ID {tag_id}")
            print("   Status:", response.status_code)
            failed += 1

    print("\n==============================")
    print("🎯 DELETE SELESAI")
    print(f"Berhasil : {deleted}")
    print(f"Gagal    : {failed}")
    print("==============================")
